# DIRAC: Plotting radial distributions

This tutorial demonstrates how to plot radial distributions using the VISUAL module in DIRAC. It is an adaptation of the older tutorial (https://www.diracprogram.org/doc/release-25/tutorials/visual/radial_distributions/tutorial.html), which investigated the valence $3s\rightarrow 3p$ excitation of the magnesium atom and investigated at what radial distances the corresponding electric dipole transition moments pick up contributions.

For the introductory notebook to visualization in DIRAC, see `visual_interactive.ipynb`, and for an overview of VISUAL module capabilities, see `visual_overview.ipynb`.

Here, we show how to integrate and/or visualize selected unperturbed and perturbed densities on grids for a few simple molecules. Note that we use precalculated densities, for which we used a poor setup for demonstration purposes. For production calculations, the computational model needs to be well-calibrated.

The calculations presented in this notebook can also be invoked by running the `visual_tutorial_radial_distributions` test (`ctest -R visual_tutorial_radial_distributions`) in the DIRAC automatic test system.

Here, we use the `py3Dmol` library and ParaView for visualization and [RDKit](https://www.rdkit.org/) for molecular data manipulation, but any visualization software for plotting volumetric data can be used. We also demonstrate how to use different file formats, including HDF5, for which the `h5py` library is required. 
Please make sure that you have all the required libraries (see the "Prerequisites" section below). 

*Tutorial adapted by Gosia Olejniczak, Warsaw University, 2025*

### Prerequisites: environment preparation

If you are using `conda` package manager, then you can create a working environment as follows:

```
cd conda_envs
conda env create -f dirac_visual.yml
cd ..
```

Make sure the environment is activated before running this notebook:

```
conda activate dirac_visual
```

If successful, you should be able to execute the next cell without errors:

In [ ]:
import os,sys
from pathlib import Path
import shutil
import subprocess
from IPython.display import display, HTML
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
Chem.Draw.IPythonConsole.ipython_3d = True
from rdkit import rdBase
from rdkit.Chem import rdmolfiles
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem import rdMolAlign
from rdkit.Chem import rdMolTransforms

import py3Dmol

from ipywidgets import interact,fixed,IntSlider

Next, we set up the working directory for all files used or generated in this notebook:

In [ ]:
notebook_path=Path().parent.resolve()
dirac_path=notebook_path.parent.resolve()
dirac_testset=Path(dirac_path, 'test').resolve()

test_name='visual_tutorial_radial_distributions'
workdir=Path(Path.cwd(), 'workdir', test_name).resolve()
workdir.mkdir(parents=True, exist_ok=True)

#### Functions used in this tutorial

In [ ]:
def from_xyz_to_mol(fxyz):
    '''read xyz file, generate RDKit mol object'''
    raw_mol = rdmolfiles.MolFromXYZFile(str(fxyz))
    mol = Chem.Mol(raw_mol)
    rdDetermineBonds.DetermineConnectivity(mol)
    return mol

def view_single_mol(mol, add_volumetric=False, isovalue=0.1, cube_file=None):
    '''interactive visualization of mol (RDKit mol object)'''
    if add_volumetric:
        # mol + surface
        data = None
        with open(cube_file, "r") as infile:
            data = infile.read()
        view=py3Dmol.view(width=400, height=300)
        view.addVolumetricData(
            data,
            "cube",
            {
                "isoval": isovalue,
                "smoothness": 5,
                "opacity": 0.8,
                "volformat": "cube",
                "color": "blue",
            },
        )
        view.addModel(data, "cube")
        view.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
        view.zoomTo()
    else:
        # only mol        
        view = py3Dmol.view(
            data=Chem.MolToMolBlock(mol),
            style={"stick": {}, "sphere": {"scale": 0.3}},
            width=300,
            height=300,
        )
    return view.show()

def dirac_output_file(dirname,finp,fmol):
    inp_string=str(finp).split('/')[-1].strip('.inp')
    mol_string=str(fmol).split('/')[-1].strip('.mol').strip('.xyz')       
    fout=Path(dirname, inp_string+'_'+mol_string+'.out').resolve()
    return str(fout)
    
def grep_from_dirac(pattern1, pattern2, file, exclude_last=False):
    if exclude_last:
        greplines="sed -n '/%s/,/^[^%s]/p' %s " %(pattern1, pattern2, file)
    else:
        greplines="sed -n '/%s/,/%s/p' %s " %(pattern1, pattern2, file)
    out=subprocess.run(greplines, shell=True, text=True, capture_output=True)
    print(out.stdout)


## Getting the wave function and transition moments

In this section, we explore various unperturbed densities of the benzene molecule. We start with setting up the working environment: 

In [ ]:
os.chdir(str(workdir))
print('Demonstration directory: %s' %(workdir))

In [ ]:
dirac_test=Path(dirac_testset, test_name).resolve()
fmol=str(Path(dirac_test, 'Mg.xyz').resolve())
finp1=str(Path(dirac_test, 'Exc.inp').resolve())
finp2=str(Path(dirac_test, 'dens.inp').resolve())
finp3=str(Path(dirac_test, 'length.inp').resolve())
finp4=str(Path(dirac_test, 'velocity.inp').resolve())
results_dir=Path(dirac_test, 'result').resolve()

fout1=str(Path(results_dir, 'Exc_Mg.out').resolve())
fout2=str(Path(results_dir, 'dens_Mg.out').resolve())
fout3=str(Path(results_dir, 'length_Mg.out').resolve())
fout4=str(Path(results_dir, 'velocity_Mg.out').resolve())

print('The input files are in the corresponding DIRAC test directory: %s' %(dirac_test))

First, we run the following calculations; we save the `CHECKPOINT.h5` and `RESPONSE.h5` files which have the information about the wave function and response vectors, respectively:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --outcmo --get="PAMXVC RESPONSE.h5"' % (finp1, fmol)
print('DIRAC run command (step 1): ')
print(cmd)

Note that `RESPONSE.h5` contains the same data as `PAMXVC`, but in hdf5 format that is used in VISUAL. Here, we export both, as inspecting the `PAMXVC` file with `rspread.x` (demonstrated below) gives a user-friendly output.

The magnesium atom is a light atom, so we expect relativistic effects, spin-orbit interaction in particular, to be quite weak, and we can assume more or less pure singlets and triplets. Single excitations $3s\rightarrow 3p$ lead to

$$
   \left(\begin{array}{c}2\\1\end{array}\right)\left(\begin{array}{c}6\\5\end{array}\right) = 12
$$

microstates, of which 9 are triplets and 3 are singlets. 

The atomic calculation will be run in the
$D_{2h}$ subgroup and we shall focus on excitations $3s\rightarrow 3p$ in the irrep $B_{3u}$, which is the symmetry of the $x$-coordinate. We start from a closed-shell reference, which is therefore totally symmetric. This means that the symmetry of the excitation is also the symmetry of the excited state. One should note that the symmetry refers to the *total*	 symmetry, including both spatial and spin symmetry. Singlets $S$ are totally symmetric, whereas the components $(T_x,T_y,T_z)$ of the triplet transform as the rotations. From further symmetry analysis we deduce that there will be three $3s\rightarrow 3p$ excitations $B_{3u}$:

i) a $3s\rightarrow 3p_y$ transition of the $T_z$ triplet component, 

ii) a $3s\rightarrow 3p_z$ transition of the $T_y$ triplet component, and

iii) a $3s\rightarrow 3p_x$ transition of singlet character. From Hund's rules we expect the triplet excitations to be the lowest one and will therefore focus on the third, singlet excitation, which is not spin-forbidden. Indeed, when we look in the output we find     

```
================================================================================

 *** Excitations of boson symmetry 2 : B3u

--------------------------------------------------------------------------------
 Excitation no.   1 excitation energy     0.04122747 a.u.    5.58E-06 (converged)
 Linear symmetry:  1u                         1.1219 eV
 *** Transition moments <0|A|n> for this excitation:

 A - xdipvel              B3u  T-         0.73226613E-04 a.u.
 A - xdiplen              B3u  T+         0.17945921E-02 a.u.
--------------------------------------------------------------------------------
 Excitation no.   2 excitation energy     0.04161588 a.u.    5.58E-06 (converged)
 Linear symmetry:  1u                         1.1324 eV
 *** Transition moments <0|A|n> for this excitation:

 A - xdipvel              B3u  T-        -0.17966513E-11 a.u.
 A - xdiplen              B3u  T+         0.22401089E-09 a.u.
--------------------------------------------------------------------------------
 Excitation no.   3 excitation energy     0.14917197 a.u.    1.65E-06 (converged)
 Linear symmetry:  1u                         4.0592 eV
 *** Transition moments <0|A|n> for this excitation:

 A - xdipvel              B3u  T-         0.358852917641 a.u.
 A - xdiplen              B3u  T+         2.403791564532 a.u.
================================================================================
```

showing that the first two excitations have essentially no oscillator strength, in contrast to the final one. 


### Plotting radial distributions

The oscillator strength is given by

$f_{fi}=\frac{2\omega}{\hbar e^2}\left|\langle f|\hat{T}(\omega_{fi})|i\rangle\right|^2$

where the excitation energy is given by $\Delta E = \hbar\omega_{fi}$. Within the electric dipole approximation the
reduced interaction operator $\hat{T}$ is given by

$\hat{T}_{L}=\boldsymbol{\mu}\cdot\boldsymbol{\epsilon};\quad \boldsymbol{\mu}=-e\mathbf{r}$,

in the length representation and

$\hat{T}_{V}=\frac{e}{\omega}c\boldsymbol{\alpha}\cdot\boldsymbol{\epsilon}$

in the velocity representation. When the vector $\boldsymbol{\epsilon}$ indicating the
(polarization) orientation of the electric field component of the incident light is set to the
Cartesian unit vector $\boldsymbol{e}_{x}$, the interaction operator span the $B_{3u}$ irrep.
These are the operators specified in the input. However, in the velocity representation the reduced
interaction operator $\hat{T}_{V}$ is frequency-dependent, and since we do not know excitation energies
before running our calculation, this part was not specified. The transition moments of the two interaction operators
are therefore strikingly different. However, is we take the transition moment of 0.358852917641 a.u. in the
velocity representation and divide by the excitation energy 0.14917197 a.u., we get 2.4056323560049524 a.u.,
which is rather close to the transition moment of 2.403791564532 a.u. obtained in the length representation.
It is close, but not identical, because we are using *finite* basis sets. The interaction operator in the two
representations may have different basis set requirements and this is what we shall investigate by looking at their
radial distribution.

As a reference we will use the number density of the atom, but we will scale all quantities such that they
integrate to one, in order to facilitate comparison. We start by getting the radial distribution for the
number density. The VISUAL part of an input (`dens.inp`) reads 

```
**VISUAL
.GRIDS
2
id=1 input=radial
id=2 input=dft
.RADIAL
0.0,  0.0,   0.0
10.0
500
.GRIDFUNCTIONS
4
name=ed id_grid=1 dscale=12.0 purpose=integration
name=ed id_grid=2 dscale=12.0 purpose=integration
name=ed id_grid=1 dscale=12.0 purpose=visualization export=hdf5 file_out=dens_radial.h5
name=ed id_grid=1 dscale=12.0 purpose=visualization export=csv file_out=dens_radial.csv
*END OF
```

where we scale the density down by the 12 electrons of magnesium. We run the job using:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --put="RESPONSE.h5" --get="dens*.csv"' % (finp2, fmol)
print('DIRAC run command: ')
print(cmd)

Next we turn to the transition moment densities. This requires a little bit more thought. Let us have a look at the file PAMXVC of solution vectors. This is an unformatted file, but each entry comes with a label which can be read by the utility program rspread.x. Assuming the program is in your path, you read this information by running:

```
rspread.x PAMXVC
```

The output is

```
 Solution vectors found on file : PAMXVC

** Solution vector : PP EXCITATIONB3u Irrep:   2 Trev:   1
  1 Type : E+ Freq.:     0.041227 Rnorm:   0.56E-05 Length:       300
  2 Type : E+ Freq.:     0.041616 Rnorm:   0.56E-05 Length:       300
  3 Type : E+ Freq.:     0.149172 Rnorm:   0.17E-05 Length:       300
  4 Type : E- Freq.:     0.041227 Rnorm:   0.56E-05 Length:       300
  5 Type : E- Freq.:     0.041616 Rnorm:   0.56E-05 Length:       300
  6 Type : E- Freq.:     0.149172 Rnorm:   0.17E-05 Length:       300
  7 Type : P+ Freq.:     0.041227 Rnorm:   0.56E-05 Length:       318
  8 Type : P+ Freq.:     0.041616 Rnorm:   0.56E-05 Length:       318
  9 Type : P+ Freq.:     0.149172 Rnorm:   0.17E-05 Length:       318
 10 Type : P- Freq.:     0.041227 Rnorm:   0.56E-05 Length:       318
 11 Type : P- Freq.:     0.041616 Rnorm:   0.56E-05 Length:       318
 12 Type : P- Freq.:     0.149172 Rnorm:   0.17E-05 Length:       318
```

Although there are 3 excitations, there are in fact 12 entries. This is because each solution vector is split into contributions corresponding to e-e rotations, and contributions corresponding to e-p rotations. Each component is further split into contributions that are time symmetric and contributions that are time antisymmetric. The transition moments are obtained by contracting solution vectors with property gradients (see [Bast2009](https://onlinelibrary.wiley.com/doi/10.1002/qua.22065) for details). A time-(anti)symmetric operator gives a non-zero contribution only by contraction with the time-(anti)symmetric part of the solution vector. The reduced interaction operator in the length and velocity representations is time-symmetric and time-antisymmetric, respectively. With this in mind we set up the following input (length.inp) for generating the radial distribution of the transition moment in the length representation - its VISUAL part reads

```
**VISUAL
.GRIDS
2
id=1 input=radial
id=2 input=dft
.RADIAL
0.0,  0.0,   0.0
10.0
500
.GRIDFUNCTIONS
8
name=edip id_grid=1 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=3 purpose=integration
name=edip id_grid=1 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=9 purpose=integration
name=edip id_grid=1 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=3 purpose=visualization export=hdf5 file_out=length_radial_rspvec3.h5
name=edip id_grid=1 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=3 purpose=visualization export=csv file_out=length_radial_rspvec3.csv
name=edip id_grid=1 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=9 purpose=visualization export=hdf5 file_out=length_radial_rspvec9.h5
name=edip id_grid=1 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=9 purpose=visualization export=csv file_out=length_radial_rspvec9.csv
name=edip id_grid=2 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=3 purpose=integration
name=edip id_grid=2 dscale=2.403791564532 rspvec_file=RESPONSE.h5 rspvec_id=9 purpose=integration
```

The indices of response vectors on the `RESPONSE.h5` file are the same as on `PAMXVC` file; hence, e.g., here we specified `rspvec_id=3` and `rspvec_id=9`, which we identified earlier from inspecting the `Exc_Mg.out` and `rspread.x PAMXVC` outputs. Alternatively, to identify an index of the response vector that should be considered in visualization directly from the `RESPONSE.h5` file, you can use the `h5dump RESPONSE.h5` command.

Then, we run the job using:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --put="RESPONSE.h5" --get="length*.csv"' % (finp3, fmol)
print('DIRAC run command: ')
print(cmd)

Using the same reasoning, the VISUAL part of an input (`velocity.inp`) for generating the radial distribution of the transition moment in the velocity representation reads

```
**VISUAL
.GRIDS
2
id=1 input=radial
id=2 input=dft
.RADIAL
0.0,  0.0,   0.0
10.0
500
.GRIDFUNCTIONS
8
name=j id_grid=1 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=6 purpose=integration
name=j id_grid=1 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=12 purpose=integration
name=j id_grid=1 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=6 purpose=visualization export=hdf5 file_out=velocity_radial_rspvec6.h5
name=j id_grid=1 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=6 purpose=visualization export=csv file_out=velocity_radial_rspvec6.csv
name=j id_grid=1 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=12 purpose=visualization export=hdf5 file_out=velocity_radial_rspvec12.h5
name=j id_grid=1 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=12 purpose=visualization export=csv file_out=velocity_radial_rspvec12.csv
name=j id_grid=2 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=6 purpose=integration
name=j id_grid=2 dscale=0.358852917641 rspvec_file=RESPONSE.h5 rspvec_id=12 purpose=integration
```

and we run the job using:

In [ ]:
cmd = 'pam --inp=%s --mol=%s --noarch --nobackup --incmo --put="RESPONSE.h5" --get="velocity*.csv"' % (finp4, fmol)
print('DIRAC run command: ')
print(cmd)

In both cases the densities are scaled such that the transition moments integrate up to +1.0. Note that for the length representation, the keyword `edip` does not include the negative electron charge. For the velocity representation, the keyword `j` asks for the densities of the x/y/z components of the current density, which does include the negative electron charge, whereas the interaction operator in the velocity operator does not. Nevertheless, the solution vectors in the PAMXVC/RESPONSE.h5 file may have random phase; therefore, the integrated transition moment densities are accurate up to the sign (also see "Good practices and gotchas" below). 

In the output of the first (length) calculation we find:

```
  Final grid function values (id=  1) integrated on grid (id=  1):
   9.85302667E-001  -1.22351747E-020   4.48418056E-020
  Final grid function values (id=  2) integrated on grid (id=  1):
   1.46876454E-010   2.14397636E-030  -9.85742920E-030
  Final grid function values (id=  5) integrated on grid (id=  2):
   9.99999992E-001   6.35736886E-001   6.35736886E-001
  Final grid function values (id=  6) integrated on grid (id=  2):
   1.46884429E-010   9.32521279E-011   9.32521285E-011
```

Note that `grid (id= 1)` corresponds to the radial grid; the integration is not very precise since the radial integration uses an even-spaced grid. To get a proper 3D integration using the (ultrafine) DFT grid, we should focus on the output of `grid (id= 2)`.

Also, to get the total values of the integrated densities, we need to sum contributions from both 'e-e' and 'e-p' response vectors; 'e-e' contributions are calculated for `grid function values (id=  1)` or `grid function values (id=  5)`, while 'e-p' contributions are calculated for `grid function values (id=  2)` or `grid function values (id=  6)`. 

Since we are interested in the x-component of the electric field, our final results are:

$9.85302667E-001 + 1.46876454E-010 \approx 0.985302667$ for the radial grid, and

$9.99999992E-001 + 1.46884429E-010 \approx 1.0$ for the ultrafine DFT grid.

We now have three formatted files containing the radial distributions on our specified grid and you may now plot them using your favourite plotting program

In [ ]:
dfs=[]

f_dens=Path(notebook_path, 'visual_data/sample/dens_radial.csv').resolve()
dfs.append(pd.read_csv(f_dens, usecols=[0,1], names=['step', 'dens'], skiprows=1))

f_length3=Path(notebook_path, 'visual_data/sample/length_radial_rspvec3.csv').resolve()
dfs.append(pd.read_csv(f_length3, usecols=[1], names=['length3'], skiprows=1))
f_length9=Path(notebook_path, 'visual_data/sample/length_radial_rspvec9.csv').resolve()
dfs.append(pd.read_csv(f_length9, usecols=[1], names=['length9'], skiprows=1))

f_velocity6=Path(notebook_path, 'visual_data/sample/velocity_radial_rspvec6.csv').resolve()
dfs.append(pd.read_csv(f_velocity6, usecols=[1], names=['velocity6'], skiprows=1))
f_velocity12=Path(notebook_path, 'visual_data/sample/velocity_radial_rspvec12.csv').resolve()
dfs.append(pd.read_csv(f_velocity12, usecols=[1], names=['velocity12'], skiprows=1))

df=pd.concat(dfs,axis=1)

In [ ]:
plt.plot(df['step'], df['dens'], markersize=3, label='dens') 
plt.plot(df['step'], df['length3']+df['length9'],  markersize=3, label='length')
plt.plot(df['step'], df['velocity6']+df['velocity12'],  markersize=3, label='velocity')

plt.xlabel('Radial step [a.u.]')
plt.ylabel('Density values')
ax = plt.gca()
ax.set_xlim([0,10])
ax.set_ylim([0,None])

plt.legend(loc='best', title='Series')
plt.show()

Compared to the number density, the transition moment densities are very diffuse and it is clear that you may need to extend your basis set with diffuse functions in order to properly describe them. We may also note that the transition moment density in the velocity representation is more compact than the one in the length representation; we can in part understand this by the fact that the interaction operator contains a Dirac $\alpha$ matrix which couple large components to the rather compact small components.

Before closing it is interesting to compare the above plot with the one obtained by
looking at core excitations $1s\rightarrow 3p$ rather than valence $3s\rightarrow 3p$ ones.
We straightforwardly obtain these excitations by specifying what occupied orbitals we may excite from in the
excitation energy section, that is:

```
*EXCITATION ENERGIES
.OCCUP
1

[...]
```

<img src="visual_data/static/tmom_rad_core.png">



The transition moment radial distributions are seen to be very much more compact (note the change of axes). We can understand this since the transition moment density now includes a compact core orbital.

### Good practices and gotchas

In both cases the densities are scaled such that the transition moments should integrate up to +1.0. However, the transition dipole moments have an arbitrary sign as the phase of solution vectors on the `RESPONSE.h5` file is random. Therefore, if your integrated values are close to -1.0, then, for plotting, change the sign of values on the generated `length*csv` and `velocity*csv` file, e.g., by:

```
df['length3']= - df['length3']
```

etc.